# 🐎 パドック歩様解析（Colab版）

動画を入れて▶を押すだけ。GPUで全頭を自動解析し、**気配ランキング＋骨格動画**を出します。

## ⚠️ 最初に必ず GPU を有効化
「ランタイム」→「ランタイムのタイプを変更」→ **GPU** → 保存

## 使い方
上から順に ▶ を押す。①は初回5分ほど。③で動画、④で解析。
**「馬番を検出できませんでした」と出たら → ③.5 で位置合わせ**をしてください。


## ① セットアップ（初回だけ・5分ほど）
出力の `✅GPU: True` を確認

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install --upgrade pip > /dev/null
!pip -q install --use-pep517 filterpy > /dev/null
!pip -q install "deeplabcut[modelzoo]" opencv-python-headless pytesseract yt-dlp tables > /dev/null
import torch
print(("✅ GPU: True " + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "❌ GPU: False → ランタイムのタイプでGPUを選び①から再実行")


## ② ツールを書き出す（一瞬）

In [ ]:
%%writefile paddock_gait.py
#!/usr/bin/env python3
"""
paddock_gait.py — パドック動画（ローカル or 動画URL）から馬の歩様指標を算出

DeepLabCut SuperAnimal-Quadruped（ゼロショット）で骨格推定し、
ストライド頻度・リズム安定性・ストライド長・頭の上下動・トップライン安定性を
体長正規化して算出、CSVに保存する。

使い方:
    # ローカル動画
    python paddock_gait.py path/to/clip.mp4
    # 動画URL（YouTube等、yt-dlp対応サイト）を渡すと自動ダウンロードして解析
    python paddock_gait.py "https://..." --start 00:12 --end 00:20
    # 区間指定はローカル動画にも使える（真横区間だけ切り出して解析）
    python paddock_gait.py clip.mp4 --start 00:03 --end 00:11
    python paddock_gait.py --selftest   # 依存なしで解析ロジックのみ検証

注意: SuperAnimalモデルは研究用途（非商用）ライセンス。内部検討に留め、
      有料コンテンツへの組み込みは規約を確認のこと。
      URLからのダウンロードは各サイトの利用規約・著作権を確認のこと。
"""
import argparse, glob, os, shutil, subprocess, sys, tempfile
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

PCUTOFF = 0.6  # この信頼度未満はNaN→補間


# ---------------- 入力取得（URL or ローカル、任意で区間切り出し） ----------------
def is_url(s):
    return s.startswith("http://") or s.startswith("https://")


def fetch_video(source, start=None, end=None, workdir="."):
    """URLならDL、ローカルならそのまま。start/end指定時は該当区間だけにする。"""
    if is_url(source):
        if shutil.which("yt-dlp") is None:
            raise RuntimeError("yt-dlp が必要です:  pip install yt-dlp")
        out_tmpl = os.path.join(workdir, "paddock_dl.%(ext)s")
        cmd = ["yt-dlp", "-f", "mp4/bestvideo+bestaudio/best",
               "--merge-output-format", "mp4", "-o", out_tmpl]
        if start and end:
            cmd += ["--download-sections", f"*{start}-{end}", "--force-keyframes-at-cuts"]
        cmd += [source]
        print(f"[0/3] 動画をダウンロード中: {source}")
        subprocess.run(cmd, check=True)
        cands = sorted(glob.glob(os.path.join(workdir, "paddock_dl.*")))
        if not cands:
            raise FileNotFoundError("ダウンロードした動画が見つかりません")
        return cands[0]

    # ローカルファイル
    if not os.path.exists(source):
        raise FileNotFoundError(source)
    if start and end:
        if shutil.which("ffmpeg") is None:
            raise RuntimeError("区間切り出しには ffmpeg が必要です")
        trimmed = os.path.splitext(source)[0] + "_trim.mp4"
        print(f"[0/3] 区間を切り出し中: {start} - {end}")
        subprocess.run(["ffmpeg", "-y", "-ss", start, "-to", end, "-i", source,
                        trimmed], check=True)
        return trimmed
    return source


# ---------------- 推論 ----------------
def downsample_fps(video_path, target_fps):
    """target_fps に間引いた動画を作って返す（高速化）。ffmpeg 必須。"""
    if shutil.which("ffmpeg") is None:
        print("警告: ffmpeg が無いため fps 間引きをスキップ")
        return video_path
    out = os.path.splitext(video_path)[0] + f"_fps{int(target_fps)}.mp4"
    subprocess.run(["ffmpeg", "-y", "-i", video_path, "-r", str(target_fps),
                    "-an", out], check=True)
    return out


def run_inference(video_path, video_adapt=True, model_name="hrnet_w32",
                  detector_name="fasterrcnn_resnet50_fpn_v2", scales=None):
    import deeplabcut, time
    scale_list = scales if scales is not None else range(200, 600, 50)
    print(f"[1/3] 骨格推定を実行: {video_path} "
          f"(pose={model_name}, detector={detector_name})")
    t0 = time.time()
    deeplabcut.video_inference_superanimal(
        [video_path],
        "superanimal_quadruped",
        model_name=model_name,
        detector_name=detector_name,
        video_adapt=video_adapt,
        scale_list=scale_list,
    )
    print(f"    推論所要時間: {time.time() - t0:.1f}s")
    stem = os.path.splitext(os.path.basename(video_path))[0]
    d = os.path.dirname(video_path) or "."
    cands = sorted(glob.glob(os.path.join(d, f"{stem}*superanimal_quadruped*.h5")))
    if not cands:
        cands = sorted(glob.glob("*superanimal_quadruped*.h5"))
    if not cands:
        raise FileNotFoundError("推論結果の.h5が見つかりません")
    return cands[-1]


def get_fps(video_path, default=30.0):
    try:
        import cv2
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or default
        cap.release()
        return float(fps)
    except Exception:
        return default


# ---------------- 解析 ----------------
def pick_best_individual(df):
    """DLC 3.0のマルチ個体出力から、最も高信頼で検出された個体を1頭選ぶ。
    未検出個体は likelihood=-1 のプレースホルダなので除外される。"""
    lik = df.xs("likelihood", axis=1, level="coords")  # (scorer, individuals) or (individuals)
    lik = lik.where(lik >= 0)  # -1 プレースホルダを無視
    score = lik.mean().groupby(level="individuals").mean()
    return score.idxmax()


def extract_target_horse(df, conf=0.35, min_kpts=6, jump_frac=0.6):
    """マルチ個体h5から『本命馬』を単一トラックとして抽出し (bodyparts, coords) を返す。

    各フレームで「高信頼キーポイントが多く・体が大きく(=手前)・前フレームに連続する」
    個体を選ぶ。さらに体中心が大きく飛ぶフレーム(別馬への乗り移り)はNaNで除外する。
    複数馬・引き馬・写り込みのある地方競馬パドックで、目的馬をロックするための処理。
    """
    import numpy as np, pandas as pd
    names = list(df.columns.names or [])
    if "individuals" not in names:
        # 単一個体: scorerを落として (bodyparts, coords) に
        while df.columns.nlevels > 2:
            df = df.droplevel(0, axis=1)
        return df
    while df.columns.nlevels > 3:      # scorerを除去 → (individuals, bodyparts, coords)
        df = df.droplevel(0, axis=1)

    inds = list(df.columns.get_level_values("individuals").unique())
    bps = list(df.columns.get_level_values("bodyparts").unique())
    n = len(df)
    cols = pd.MultiIndex.from_product([bps, ["x", "y", "likelihood"]],
                                      names=["bodyparts", "coords"])
    colidx = {c: j for j, c in enumerate(cols)}
    out = np.full((n, len(cols)), np.nan)
    cens = np.full((n, 2), np.nan)
    spans = np.full(n, np.nan)
    prev = None
    for i in range(n):
        row = df.iloc[i]
        best = None  # (score, pts, cen, span)
        for a in inds:
            xs, ys, pts = [], [], {}
            for bp in bps:
                lk = row.get((a, bp, "likelihood"))
                if lk is not None and lk == lk and lk >= conf:
                    x = row.get((a, bp, "x")); y = row.get((a, bp, "y"))
                    if x == x and y == y:
                        pts[bp] = (x, y, lk); xs.append(x); ys.append(y)
            if len(pts) < min_kpts:
                continue
            span = ((max(xs) - min(xs)) ** 2 + (max(ys) - min(ys)) ** 2) ** 0.5
            cen = (sum(xs) / len(xs), sum(ys) / len(ys))
            score = span  # 大きい(手前)ほど本命
            if prev is not None:  # 前フレームに近いほど加点(連続性)
                score -= 0.7 * ((cen[0] - prev[0]) ** 2 + (cen[1] - prev[1]) ** 2) ** 0.5
            if best is None or score > best[0]:
                best = (score, pts, cen, span)
        if best is not None:
            _, pts, cen, span = best
            for bp, (x, y, lk) in pts.items():
                out[i, colidx[(bp, "x")]] = x
                out[i, colidx[(bp, "y")]] = y
                out[i, colidx[(bp, "likelihood")]] = lk
            cens[i] = cen; spans[i] = span; prev = cen
    # 体中心が中央値から大きく外れるフレーム=別馬への乗り移りとみなし除外
    valid = ~np.isnan(cens[:, 0])
    if valid.sum() >= 5:
        med = np.nanmedian(cens[valid], axis=0)
        thr = max(np.nanmedian(spans[valid]) * jump_frac, 40.0)
        for i in range(n):
            if valid[i] and ((cens[i, 0] - med[0]) ** 2 + (cens[i, 1] - med[1]) ** 2) ** 0.5 > thr:
                out[i, :] = np.nan
    return pd.DataFrame(out, columns=cols)


def analyze_h5(df, fps):
    # 目的馬ロック: 毎フレーム最大・手前・連続する馬を選び、乗り移りフレームを除外
    df = extract_target_horse(df)

    def lik_mean(bp):
        return df[(bp, "likelihood")].mean()

    def coord(bp, c):
        s = df[(bp, c)].copy()
        s[df[(bp, "likelihood")] < PCUTOFF] = np.nan
        return s.interpolate(limit_direction="both")

    front = "front_right_paw" if lik_mean("front_right_paw") >= lik_mean("front_left_paw") else "front_left_paw"
    back = "back_right_paw" if lik_mean("back_right_paw") >= lik_mean("back_left_paw") else "back_left_paw"

    body_len = np.nanmedian(np.hypot(coord("neck_base", "x") - coord("tail_base", "x"),
                                     coord("neck_base", "y") - coord("tail_base", "y")))

    rel_x = coord(front, "x") - coord("back_base", "x")
    rel_x = (rel_x - rel_x.mean()).rolling(max(1, int(fps * 0.1)), center=True, min_periods=1).mean()
    amp = rel_x.std()
    peaks, _ = find_peaks(rel_x.values, distance=max(1, int(fps * 0.35)), prominence=amp * 0.5)
    intervals = np.diff(peaks) / fps
    stride_freq = float(1 / np.mean(intervals)) if len(intervals) else float("nan")
    regularity = float(np.std(intervals)) if len(intervals) else float("nan")

    paw_exc = coord(front, "x") - coord("back_base", "x")
    stride_len_norm = float((np.nanpercentile(paw_exc, 95) - np.nanpercentile(paw_exc, 5)) / body_len)
    nose_y = coord("nose", "y")
    head_bob = float((np.nanpercentile(nose_y, 95) - np.nanpercentile(nose_y, 5)) / body_len)
    topline = float(np.mean([coord(b, "y").std() / body_len for b in ["back_base", "back_middle", "tail_base"]]))

    return {
        "n_strides": int(len(peaks)),
        "stride_freq_hz": round(stride_freq, 3),
        "stride_regularity_s": round(regularity, 4),
        "stride_len_norm": round(stride_len_norm, 3),
        "head_bob_norm": round(head_bob, 4),
        "topline_stability": round(topline, 4),
        "near_front": front,
        "near_back": back,
        "body_len_px": round(float(body_len), 1),
    }


def selftest():
    np.random.seed(0)
    fps, n = 60, 300
    t = np.arange(n) / fps
    scorer = "DLC_superanimal_quadruped_hrnetw32"
    bps = ["nose", "neck_base", "back_base", "back_middle", "tail_base",
           "front_left_paw", "front_right_paw", "back_left_paw", "back_right_paw"]
    cols = pd.MultiIndex.from_product([[scorer], bps, ["x", "y", "likelihood"]],
                                     names=["scorer", "bodyparts", "coords"])
    df = pd.DataFrame(np.zeros((n, len(cols))), columns=cols)
    base_x = {"nose": 500, "neck_base": 420, "back_base": 300, "back_middle": 260, "tail_base": 180,
              "front_left_paw": 410, "front_right_paw": 415, "back_left_paw": 210, "back_right_paw": 215}
    base_y = {"nose": 180, "neck_base": 150, "back_base": 140, "back_middle": 142, "tail_base": 150,
              "front_left_paw": 330, "front_right_paw": 330, "back_left_paw": 330, "back_right_paw": 330}
    hz = 1.2
    for bp in bps:
        x = base_x[bp] + 40 * t
        y = np.full(n, base_y[bp], float)
        phase = 0 if "right" in bp else np.pi
        if "paw" in bp:
            x = x + 25 * np.sin(2 * np.pi * hz * t + phase)
            y = y - 20 * np.clip(np.sin(2 * np.pi * hz * t + phase), 0, None)
        if bp == "nose":
            y = y + 8 * np.sin(2 * np.pi * hz * t)
        df[(scorer, bp, "x")] = x + np.random.normal(0, 1, n)
        df[(scorer, bp, "y")] = y + np.random.normal(0, 1, n)
        df[(scorer, bp, "likelihood")] = 0.4 if "left" in bp else 0.95
    m = analyze_h5(df, fps)
    print("selftest metrics:", m)
    assert m["near_front"] == "front_right_paw"
    assert abs(m["stride_freq_hz"] - hz) < 0.2, m["stride_freq_hz"]
    print("SELFTEST PASSED (stride_freq %.3f ≈ %.1f)" % (m["stride_freq_hz"], hz))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("video", nargs="?", help="パドック動画のパス、または動画URL")
    ap.add_argument("--start", help="切り出し開始 (例 00:12) URL/ローカル両対応")
    ap.add_argument("--end", help="切り出し終了 (例 00:20)")
    ap.add_argument("--no-adapt", action="store_true", help="video_adaptを切って高速化")
    ap.add_argument("--fps", type=float, help="この fps に間引いて高速化 (例 12)。歩様は約1Hzなので10〜15で十分")
    ap.add_argument("--fast", action="store_true",
                    help="軽量モデル(mobilenet検出器+rtmpose_s, 単一スケール)で高速化。精度は少し落ちる")
    ap.add_argument("--model", help="ポーズモデル上書き (hrnet_w32/resnet_50/rtmpose_s)")
    ap.add_argument("--detector", help="検出器上書き (fasterrcnn_resnet50_fpn_v2/fasterrcnn_mobilenet_v3_large_fpn/ssdlite)")
    ap.add_argument("--selftest", action="store_true", help="解析ロジックのみ検証")
    args = ap.parse_args()

    if args.selftest:
        selftest(); return
    if not args.video:
        ap.error("動画パスまたはURLを指定してください（または --selftest）")

    video = fetch_video(args.video, args.start, args.end, workdir=".")
    if args.fps:
        src_fps = get_fps(video)
        if args.fps < src_fps:
            print(f"[0/3] fpsを{src_fps:.0f}→{args.fps:g}に間引き（高速化）")
            video = downsample_fps(video, args.fps)
    fps = get_fps(video)
    print(f"FPS: {fps}")
    model_name = args.model or ("rtmpose_s" if args.fast else "hrnet_w32")
    detector_name = args.detector or (
        "fasterrcnn_mobilenet_v3_large_fpn" if args.fast else "fasterrcnn_resnet50_fpn_v2")
    scales = [400] if args.fast else None
    h5_path = run_inference(video, video_adapt=not args.no_adapt,
                            model_name=model_name, detector_name=detector_name, scales=scales)
    print(f"[2/3] 結果読み込み: {h5_path}")
    df = pd.read_hdf(h5_path)
    print(f"[3/3] 歩様指標を算出")
    m = analyze_h5(df, fps)
    m["source"] = args.video
    m["fps"] = fps

    for k in ["n_strides", "stride_freq_hz", "stride_regularity_s",
              "stride_len_norm", "head_bob_norm", "topline_stability",
              "near_front", "near_back", "body_len_px"]:
        print(f"  {k:22s}: {m[k]}")

    out_csv = os.path.splitext(video)[0] + "_gait_metrics.csv"
    pd.DataFrame([m]).to_csv(out_csv, index=False)
    print(f"\n保存しました: {out_csv}")
    labeled = sorted(glob.glob(os.path.splitext(os.path.basename(video))[0] + "*superanimal_quadruped*.mp4"))
    if labeled:
        print(f"骨格付き動画（要目視確認）: {labeled[-1]}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paddock_segment.py
#!/usr/bin/env python3
"""
paddock_segment.py — パドック連続映像を「馬番テロップOCR」で馬ごとに自動分割

地方競馬(NAR)のパドック中継は全頭が1本の映像で流れる。画面下部の
馬番テロップをOCRで読み取り、馬番が変わったタイミングで区間を切り出す。
切り出した各区間に paddock_gait.py を掛ければ、1レース全頭の歩様指標を
自動で取得できる。

使い方:
    # 馬番だけ検出して区間一覧を表示
    python paddock_segment.py race.mp4
    # 各馬のクリップを切り出す
    python paddock_segment.py race.mp4 --cut
    # 切り出し＋各馬の歩様解析まで一気に
    python paddock_segment.py race.mp4 --cut --gait --fast --fps 12
    # テロップ位置が違う競馬場向けに馬番ボックス座標を指定
    python paddock_segment.py race.mp4 --roi 258,806,120,70
    python paddock_segment.py --selftest

注意: 馬番ボックスの座標(--roi)は競馬場・配信レイアウトごとに調整が必要。
      既定値は高知けいばナイター配信(1920x1140)向け。
"""
import argparse, json, os, subprocess, sys


# 高知けいば配信(1920x1140)の馬番テロップ既定座標 (x, y, w, h)
DEFAULT_ROI = (258, 806, 120, 70)


def read_umaban(roi_bgr, psms=(10, 8, 7, 13)):
    """馬番ボックス画像から数字をOCR。読めなければ None。
    中抜き(アウトライン)フォントに対応するため穴埋め処理を行う。
    psms を絞ると高速化(位置の自動探索時は単一psmで走査する)。"""
    import cv2, numpy as np, pytesseract
    g = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
    _, th = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    inv = cv2.bitwise_not(th)
    inv = cv2.morphologyEx(inv, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    ff = inv.copy()
    h, w = inv.shape
    mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(ff, mask, (0, 0), 255)
    filled = inv | cv2.bitwise_not(ff)          # 数字を白塗り(黒地)
    out = cv2.bitwise_not(filled)               # 黒数字・白地(tesseract向き)
    out = cv2.copyMakeBorder(out, 25, 25, 25, 25, cv2.BORDER_CONSTANT, value=255)
    for psm in psms:
        t = pytesseract.image_to_string(
            out, config=f"--psm {psm} -c tessedit_char_whitelist=0123456789").strip()
        if t.isdigit() and 1 <= int(t) <= 18:
            return int(t)
    return None


def auto_find_roi(video, n_scan=3):
    """馬番テロップの位置(ROI)を自動探索する。競馬場・解像度・録画枠が違っても
    合わせ直す手間を無くすため、画面下部・左寄りを走査し、時間で変化する数字が
    最もよく読める位置を返す。見つからなければ None。"""
    import cv2
    cap = cv2.VideoCapture(video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = []
    for f in (0.2, 0.5, 0.8)[:n_scan]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * f) if total else 0)
        ok, fr = cap.read()
        if ok:
            frames.append(fr)
    cap.release()
    if not frames:
        return None
    g0 = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    bw = max(44, int(W * 0.065)); bh = max(30, int(H * 0.055))
    # 馬番テロップは経験上、左下(x≈0.08〜0.24W, y≈0.62〜0.86H)に集中。粗く走査し上限を設ける。
    xs = list(range(int(W * 0.06), int(W * 0.26), max(14, int(W * 0.035))))
    ys = list(range(int(H * 0.62), int(H * 0.86), max(12, int(H * 0.04))))
    cands = []
    for y in ys:
        for x in xs:
            # 事前フィルタ: 馬番ボックスは「明るい地＋濃い数字」で高コントラスト。
            win = g0[y:y + bh, x:x + bw]
            if win.size == 0 or win.max() < 180 or (int(win.max()) - int(win.min())) < 90:
                continue
            if read_umaban(frames[0][y:y + bh, x:x + bw], psms=(10,)) is not None:
                cands.append((x, y))
        if len(cands) >= 12:      # 上限(暴走防止)
            break
    best, best_score = None, 0
    for x, y in cands:
        reads = [read_umaban(fr[y:y + bh, x:x + bw], psms=(10, 8)) for fr in frames]
        valid = [r for r in reads if r is not None]
        if len(valid) >= max(2, len(frames) // 2):
            score = len(valid) + len(set(valid))
            if score > best_score:
                best_score, best = score, (x, y, bw, bh)
    return best


def scan_umaban(video, roi, sample_fps):
    """動画をsample_fps間隔でOCRし、[(t_sec, umaban or None), ...] を返す。"""
    import cv2
    x, y, w, h = roi
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(1, int(round(fps / sample_fps)))
    samples = []
    fno = 0
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fno)
        ok, fr = cap.read()
        if not ok:
            break
        roi_bgr = fr[y:y + h, x:x + w]
        num = read_umaban(roi_bgr) if roi_bgr.size else None
        samples.append((fno / fps, num))
        fno += step
        if total and fno >= total:
            break
    cap.release()
    return samples, fps


def smooth(samples, win=2):
    """欠測/誤読を近傍多数決で補間したラベル列を返す。"""
    labels = [n for _, n in samples]
    out = []
    for i in range(len(labels)):
        lo, hi = max(0, i - win), min(len(labels), i + win + 1)
        votes = {}
        for n in labels[lo:hi]:
            if n is not None:
                votes[n] = votes.get(n, 0) + 1
        out.append(max(votes, key=votes.get) if votes else None)
    return out


def segment(samples, labels, min_seg, merge_gap):
    """同一馬番の連続区間を[(umaban, start, end)]にまとめる。"""
    segs = []
    cur, start = None, None
    for (t, _), lab in zip(samples, labels):
        if lab != cur:
            if cur is not None and start is not None:
                segs.append([cur, start, t])
            cur, start = lab, t
    if cur is not None and start is not None:
        segs.append([cur, start, samples[-1][0]])
    segs = [s for s in segs if s[0] is not None]
    # 同一馬番で近接する区間を結合（一時的な誤読で割れた場合の対策）
    merged = []
    for s in segs:
        if merged and merged[-1][0] == s[0] and s[1] - merged[-1][2] <= merge_gap:
            merged[-1][2] = s[2]
        else:
            merged.append(s)
    return [s for s in merged if s[2] - s[1] >= min_seg]


def fmt(t):
    return f"{int(t // 60):02d}:{t % 60:05.2f}"


def process(video, roi, sample_fps, min_seg, merge_gap, cut, gait, gait_args):
    print(f"[1/2] 馬番テロップをOCR中 (roi={roi}, sample={sample_fps}fps)…")
    samples, fps = scan_umaban(video, roi, sample_fps)
    labels = smooth(samples)
    segs = segment(samples, labels, min_seg, merge_gap)
    print(f"[2/2] {len(segs)} 区間を検出:")
    stem = os.path.splitext(video)[0]
    result = []
    for i, (uma, st, en) in enumerate(segs, 1):
        print(f"  #{uma:>2}  {fmt(st)} - {fmt(en)}  ({en - st:4.1f}s)")
        rec = {"umaban": uma, "start": round(st, 2), "end": round(en, 2),
               "dur": round(en - st, 2)}
        if cut:
            clip = f"{stem}_uma{uma:02d}.mp4"
            subprocess.run(["ffmpeg", "-y", "-ss", f"{st:.2f}", "-to", f"{en:.2f}",
                            "-i", video, "-an", clip],
                           check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            rec["clip"] = clip
            if gait:
                cmd = [sys.executable, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                       "paddock_gait.py"), clip, "--no-adapt"] + gait_args
                print(f"      → 歩様解析: {' '.join(cmd[-3:])}")
                subprocess.run(cmd, check=False)
                csv = os.path.splitext(clip)[0] + "_gait_metrics.csv"
                rec["gait_csv"] = csv if os.path.exists(csv) else None
        result.append(rec)
    out_json = f"{stem}_segments.json"
    with open(out_json, "w") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"\n区間情報を保存: {out_json}")
    return result


def selftest():
    samples = [(i * 0.5, n) for i, n in enumerate(
        [None, 3, 3, 3, None, 3, 3, 7, 7, 7, 7, None, 7, 12, 12, 12, 12])]
    labels = smooth(samples)
    segs = segment(samples, labels, min_seg=1.0, merge_gap=1.0)
    got = [s[0] for s in segs]
    print("segments:", [(u, round(a, 1), round(b, 1)) for u, a, b in segs])
    assert got == [3, 7, 12], got
    print("SELFTEST PASSED (3頭 [3,7,12] を正しく分割)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("video", nargs="?", help="パドック連続映像のパス")
    ap.add_argument("--roi", help="馬番ボックス座標 x,y,w,h（既定:高知配信）")
    ap.add_argument("--sample-fps", type=float, default=3.0, help="OCRサンプリング間隔(fps)")
    ap.add_argument("--min-seg", type=float, default=2.0, help="この秒数未満の区間は無視")
    ap.add_argument("--merge-gap", type=float, default=1.5, help="同一馬番の近接区間を結合する猶予(秒)")
    ap.add_argument("--cut", action="store_true", help="各馬のクリップを切り出す")
    ap.add_argument("--gait", action="store_true", help="各クリップに paddock_gait.py を実行")
    ap.add_argument("--selftest", action="store_true", help="分割ロジックのみ検証")
    args, gait_args = ap.parse_known_args()

    if args.selftest:
        selftest(); return
    if not args.video:
        ap.error("パドック映像のパスを指定してください（または --selftest）")

    if args.roi:
        roi = tuple(int(v) for v in args.roi.split(","))
        if len(roi) != 4:
            ap.error("--roi は x,y,w,h の4値で指定してください")
    else:
        # ROI未指定なら自動探索(競馬場・解像度が違っても合わせ直し不要)
        print("[0/2] 馬番テロップの位置を自動探索中…")
        roi = auto_find_roi(args.video)
        if roi:
            print(f"  → 自動検出したROI: {roi}")
        else:
            roi = DEFAULT_ROI
            print(f"  → 自動検出できず。既定ROI {roi} を使用（合わなければ --roi で指定）")
    process(args.video, roi, args.sample_fps, args.min_seg, args.merge_gap,
            args.cut, args.gait, gait_args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile skeleton_overlay.py
#!/usr/bin/env python3
"""
skeleton_overlay.py — 推論結果(h5)から「関節点＋骨格線」を動画に描画

DeepLabCut既定の出力は点主体なので、Tom Wilson系のウォーキング動画のように
点を線で結んだ骨格オーバーレイを描く。馬の動き（歩様)を目視検証しやすくする。

使い方:
    python skeleton_overlay.py clip.mp4                 # 同名h5を自動検出
    python skeleton_overlay.py clip.mp4 --h5 result.h5 --out skel.mp4
    python skeleton_overlay.py --selftest

出力はH.264(どこでも再生可)。信頼度 pcutoff 未満の点/線は描かない。
"""
import argparse, glob, os, subprocess, sys

PCUTOFF = 0.6

# 背骨(トップライン): き甲→背中→尾の付け根。歩様評価で最重要のため専用に強調描画する。
# ※ tail_end(尾の先)は垂れ下がるので含めない。neck_end(首)も背骨ではないので除く。
SPINE = ["neck_base", "back_base", "back_middle", "back_end", "tail_base"]

# SuperAnimal-Quadruped(39点)に対する馬の骨格線定義
SKELETON = [
    # 顔
    ("nose", "upper_jaw"), ("nose", "lower_jaw"),
    ("upper_jaw", "mouth_end_right"), ("upper_jaw", "mouth_end_left"),
    ("nose", "right_eye"), ("nose", "left_eye"),
    ("right_eye", "right_earbase"), ("left_eye", "left_earbase"),
    ("right_earbase", "right_earend"), ("left_earbase", "left_earend"),
    ("lower_jaw", "throat_base"),
    # 首→背骨→尾
    ("throat_base", "throat_end"),
    ("right_earbase", "neck_end"), ("left_earbase", "neck_end"),
    ("neck_end", "neck_base"), ("neck_base", "back_base"),
    ("back_base", "back_middle"), ("back_middle", "back_end"),
    ("back_end", "tail_base"), ("tail_base", "tail_end"),
    # 前肢(左右)
    ("neck_base", "front_left_thai"), ("neck_base", "front_right_thai"),
    ("body_middle_left", "front_left_thai"), ("body_middle_right", "front_right_thai"),
    ("front_left_thai", "front_left_knee"), ("front_left_knee", "front_left_paw"),
    ("front_right_thai", "front_right_knee"), ("front_right_knee", "front_right_paw"),
    # 後肢(左右)
    ("back_end", "back_left_thai"), ("back_end", "back_right_thai"),
    ("tail_base", "back_left_thai"), ("tail_base", "back_right_thai"),
    ("back_left_thai", "back_left_knee"), ("back_left_knee", "back_left_paw"),
    ("back_right_thai", "back_right_knee"), ("back_right_knee", "back_right_paw"),
    # 胴(腹)
    ("neck_base", "body_middle_left"), ("neck_base", "body_middle_right"),
    ("body_middle_left", "belly_bottom"), ("body_middle_right", "belly_bottom"),
    ("belly_bottom", "back_middle"), ("belly_bottom", "back_end"),
]


def cog_point(getp):
    """馬の重心(第12〜13肋骨=ゼッケン下・肘の後ろ)を推定して座標を返す。

    首(neck_base)は頭と動くので使わず、胴の点 back_base→tail_base を基準にする:
      ・水平: back_base から尾へ約28%(第12〜13肋骨あたり)
      ・垂直: その点から背骨に垂直に、体長の約22%だけ腹側へ下ろす(=バレル中ほど)
    胴基準なので斜め向きでも前へずれにくい。必要点が無ければ None。"""
    front = getp("back_base") or getp("neck_base")
    rear = getp("tail_base") or getp("back_end") or getp("back_middle")
    if not (front and rear):
        return None
    vx, vy = rear[0] - front[0], rear[1] - front[1]
    L = (vx * vx + vy * vy) ** 0.5 or 1.0
    fx, fy = front[0] + 0.52 * vx, front[1] + 0.52 * vy   # 背中の中央〜やや後ろ(重心の上)
    px, py = -vy, vx                                       # 背骨に垂直
    if py < 0:                                             # 腹方向(画面下:y+)に向ける
        px, py = vy, -vx
    depth = 0.22 * L                                       # バレル中ほどまで下ろす
    return (int(fx + depth * px / L), int(fy + depth * py / L))


def _prep_df(df):
    """マルチ個体h5から『目的馬』を単一トラックとして抽出し (bodyparts, coords) にする。
    paddock_gait.extract_target_horse と同じロジックで、毎フレーム最大・手前・連続する
    馬を選び、別馬への乗り移りフレームを除外する。"""
    import paddock_gait
    return paddock_gait.extract_target_horse(df)


def render(video, h5, out, pcutoff=PCUTOFF):
    import cv2, numpy as np, pandas as pd
    df = _prep_df(pd.read_hdf(h5))
    bps = list(df.columns.get_level_values(0).unique())
    # 各bodypartに固定色(虹)を割当
    cmap = {}
    for i, bp in enumerate(bps):
        hue = int(179 * i / max(1, len(bps) - 1))
        c = cv2.cvtColor(np.uint8([[[hue, 220, 255]]]), cv2.COLOR_HSV2BGR)[0, 0]
        cmap[bp] = (int(c[0]), int(c[1]), int(c[2]))

    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tmp = os.path.splitext(out)[0] + "_tmp.mp4"
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    def pt(row, bp):
        if (bp, "likelihood") not in df.columns:
            return None
        if row[(bp, "likelihood")] < pcutoff:
            return None
        x, y = row[(bp, "x")], row[(bp, "y")]
        if x != x or y != y:
            return None
        return (int(x), int(y))

    n = min(len(df), int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or len(df)))
    for i in range(n):
        ok, fr = cap.read()
        if not ok:
            break
        row = df.iloc[i]
        # 骨格線(黒・やや細)
        for a, b in SKELETON:
            pa, pb = pt(row, a), pt(row, b)
            if pa and pb:
                cv2.line(fr, pa, pb, (20, 20, 20), 2, cv2.LINE_AA)
        # ★背骨(トップライン): 最重要。太い緑線で強調描画。
        # 背骨はほぼ直線なので、首の根本(neck_base)と尾(tail_base)を直線1本で結ぶ。
        # 途中の背中の点はサドル/腹に落ちやすく線を凹ませるため使わない。
        avail = [p for p in (pt(row, bp) for bp in SPINE) if p]
        if len(avail) >= 2:
            a, b = avail[0], avail[-1]      # 端＝首の根本 と 尾の付け根
            cv2.line(fr, a, b, (255, 255, 255), 8, cv2.LINE_AA)   # 白フチ
            cv2.line(fr, a, b, (60, 220, 30), 4, cv2.LINE_AA)     # 緑本体
        # 関節点(色付き)。背骨の関節は大きめに
        for bp in bps:
            p = pt(row, bp)
            if p:
                r = 9 if bp in SPINE else 6
                cv2.circle(fr, p, r, cmap[bp], -1, cv2.LINE_AA)
        # ●馬の重心(第12〜13肋骨・肘の後ろ)を赤い点で表示
        cog = cog_point(lambda bp: pt(row, bp))
        if cog:
            cv2.circle(fr, cog, 13, (255, 255, 255), -1, cv2.LINE_AA)  # 白フチ
            cv2.circle(fr, cog, 10, (0, 0, 255), -1, cv2.LINE_AA)      # 赤(重心)
        vw.write(fr)
    cap.release(); vw.release()

    # H.264へ変換(どこでも再生可)
    subprocess.run(["ffmpeg", "-y", "-i", tmp, "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-crf", "23", "-preset", "veryfast", "-movflags", "+faststart", out],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.remove(tmp)
    print(f"骨格線オーバーレイ動画を保存: {out}")
    return out


def find_h5(video):
    stem = os.path.splitext(os.path.basename(video))[0]
    d = os.path.dirname(video) or "."
    c = sorted(glob.glob(os.path.join(d, f"{stem}*superanimal_quadruped*.h5")))
    return c[-1] if c else None


def selftest():
    # 骨格定義が既知bodypartのみを参照しているか検証
    known = {
        "nose","upper_jaw","lower_jaw","mouth_end_right","mouth_end_left","right_eye",
        "right_earbase","right_earend","left_eye","left_earbase","left_earend","neck_base",
        "neck_end","throat_base","throat_end","back_base","back_end","back_middle",
        "tail_base","tail_end","front_left_thai","front_left_knee","front_left_paw",
        "front_right_thai","front_right_knee","front_right_paw","back_left_paw",
        "back_left_thai","back_right_thai","back_left_knee","back_right_knee",
        "back_right_paw","belly_bottom","body_middle_right","body_middle_left",
    }
    bad = {p for e in SKELETON for p in e if p not in known}
    assert not bad, f"未知のbodypart: {bad}"
    print(f"SELFTEST PASSED (骨格線 {len(SKELETON)}本, 全て既知の関節を参照)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("video", nargs="?")
    ap.add_argument("--h5", help="推論結果h5(省略時は自動検出)")
    ap.add_argument("--out", help="出力mp4(省略時は <video>_skeleton.mp4)")
    ap.add_argument("--pcutoff", type=float, default=PCUTOFF)
    ap.add_argument("--selftest", action="store_true")
    args = ap.parse_args()
    if args.selftest:
        selftest(); return
    if not args.video:
        ap.error("動画パスを指定してください（または --selftest）")
    h5 = args.h5 or find_h5(args.video)
    if not h5 or not os.path.exists(h5):
        ap.error("推論結果h5が見つかりません。--h5 で指定してください")
    out = args.out or (os.path.splitext(args.video)[0] + "_skeleton.mp4")
    render(args.video, h5, out, args.pcutoff)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile paddock_compare.py
#!/usr/bin/env python3
"""
paddock_compare.py — 全頭の歩様指標を横並び比較・ランキング表示

paddock_segment.py --gait で出力される各馬の *_gait_metrics.csv を集め、
1レース分を横並びの比較表にする。指標ごとに相対順位(field内の位置)を付け、
「気配比較」をひと目で分かるようにする。

使い方:
    # カレントの *_gait_metrics.csv を全部集めて比較
    python paddock_compare.py
    # ディレクトリ/glob指定
    python paddock_compare.py "race1/*_gait_metrics.csv"
    # HTML表も出力（ブラウザで見やすい・相対ヒートマップ付き）
    python paddock_compare.py --html race1_compare.html
    python paddock_compare.py --selftest

注意: 指標の「良し悪し」は馬・状況で変わる。本ツールは相対順位を示すだけで、
      勝ち負けを断定しない。矢印は「一般に好まれる向き」の目安。
"""
import argparse, glob, os, sys

# 指標の定義: (キー, 表示名, 好ましい向き) direction: -1=小さいほど良い, +1=大きいほど良い, 0=中立
METRICS = [
    ("stride_freq_hz",      "ストライド頻度",   0),
    ("stride_regularity_s", "リズム安定(小=安定)", -1),
    ("stride_len_norm",     "ストライド長",     +1),
    ("head_bob_norm",       "頭の上下動(小=落着)", -1),
    ("topline_stability",   "背中の安定(小=安定)", -1),
]


def load_rows(patterns):
    """CSV群を読み、[{umaban, source, metric...}] を返す。"""
    import pandas as pd
    files = []
    for p in patterns:
        files += glob.glob(p)
    files = sorted(set(files))
    rows = []
    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception:
            continue
        if df.empty:
            continue
        r = df.iloc[0].to_dict()
        r["_file"] = f
        r["umaban"] = _guess_umaban(f, r)
        rows.append(r)
    return rows


def _guess_umaban(path, row):
    import re
    m = re.search(r"uma(\d+)", os.path.basename(path))
    if m:
        return int(m.group(1))
    src = str(row.get("source", ""))
    m = re.search(r"uma(\d+)", src)
    return int(m.group(1)) if m else None


def rank_field(rows):
    """指標ごとに field 内順位(1=好ましい向きで最上位)を付与。"""
    import pandas as pd, numpy as np
    df = pd.DataFrame(rows)
    for key, _, direction in METRICS:
        if key not in df:
            df[key] = np.nan
        vals = pd.to_numeric(df[key], errors="coerce")
        if direction == 0:
            df[key + "_rank"] = np.nan  # 中立指標は順位を付けない
        else:
            asc = (direction < 0)  # 小さいほど良い→昇順で1位
            df[key + "_rank"] = vals.rank(ascending=asc, method="min")
    return df


# 気配スコアの重み。青木修「競走馬の走りと重心」＋実結果(6番が最長ストライドで1着)を
# 反映して再較正。推進力=ストライドの伸び(後肢の踏み込み)を主役にする。
# ※head_bob(頭の上下動)は除外: 常歩で頭は横8の字に動くのが正常＝自然な頭の動きは良い。
#   将来は「頭の左右非対称」で跛行を検出して減点する予定(magnitudeだけでは判断しない)。
KEHAI_WEIGHTS = {
    "stride_len_norm": 2.0,        # ストライドの伸び=後肢の踏み込み・推進力(最重要・勝ち馬の武器)
    "topline_stability": 0.5,      # 背の安定(補助的。力強く"うねる"背も良いので過重にしない)
    "stride_regularity_s": 0.3,    # リズム(補助的。無理に揃えるのは良くない)
}


def add_kehai(df):
    """各馬の指標をfield内で0〜1に正規化し、好ましい向きに揃えて重み付き平均→
    気配スコア(0〜100, 高いほど良い)を付与する。相対比較。"""
    import numpy as np, pandas as pd
    n = len(df)
    score = pd.Series(0.0, index=df.index)
    wsum = pd.Series(0.0, index=df.index)
    for key, _, direction in METRICS:
        if direction == 0 or key not in KEHAI_WEIGHTS or key not in df:
            continue
        vals = pd.to_numeric(df[key], errors="coerce")
        vmin, vmax = vals.min(), vals.max()
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax == vmin:
            good = pd.Series(0.5, index=df.index)   # 差が無い/1頭なら中立
        else:
            good = (vals - vmin) / (vmax - vmin)
            if direction < 0:                        # 小さいほど良い指標は反転
                good = 1 - good
        w = KEHAI_WEIGHTS[key]
        m = good.notna()
        score[m] += good[m] * w
        wsum[m] += w
    df["kehai_score"] = (100 * score / wsum.replace(0, np.nan)).round(1)
    return df


def print_table(df):
    df = df.sort_values("kehai_score", ascending=False) if "kehai_score" in df else df
    print("\n=== 全頭 歩様指標 横並び比較（気配スコア順） ===")
    header = f"{'馬番':>4} | {'気配':>6} | " + " | ".join(f"{name:>14}" for _, name, _ in METRICS)
    print(header)
    print("-" * len(header))
    for _, r in df.iterrows():
        uma = r.get("umaban")
        cells = []
        for key, _, direction in METRICS:
            v = r.get(key)
            rk = r.get(key + "_rank")
            vs = "  -  " if v is None or (isinstance(v, float) and v != v) else f"{float(v):.3f}"
            mark = ""
            if direction != 0 and rk == rk and rk == 1:
                mark = "★"  # field内トップ(好ましい向き)
            cells.append(f"{vs:>13}{mark:1}")
        umas = f"{int(uma):>4}" if uma == uma and uma is not None else "  ？"
        ks = r.get("kehai_score")
        kss = "  -  " if ks is None or (isinstance(ks, float) and ks != ks) else f"{float(ks):5.1f}"
        print(f"{umas} | {kss:>6} | " + " | ".join(cells))
    print("\n気配 = 各指標を良し悪しの向きで合成した相対スコア(0〜100・高いほど良い)")
    print("★ = その指標でfield内トップ（矢印の向きで好ましい側）")
    print("※ 相対比較です。勝敗の断定ではありません。骨格付き動画での目視確認と併用してください。")


def write_html(df, path):
    import pandas as pd, numpy as np
    df = df.sort_values("kehai_score", ascending=False) if "kehai_score" in df else df
    def cell(v, rk, direction, n):
        if v is None or (isinstance(v, float) and v != v):
            return '<td style="text-align:center;color:#999">-</td>'
        bg = "#ffffff"
        if direction != 0 and rk == rk and n > 1:
            t = 1 - (rk - 1) / (n - 1)      # 1=好ましい向きの上位
            g = int(200 + 55 * t); rr = int(255 - 90 * t); bb = int(255 - 90 * t)
            bg = f"rgb({rr},{g},{bb})"
        star = "★" if (direction != 0 and rk == 1) else ""
        return f'<td style="text-align:right;background:{bg};padding:6px 10px">{float(v):.3f} {star}</td>'
    def kcell(ks):
        if ks is None or (isinstance(ks, float) and ks != ks):
            return '<td style="text-align:center;color:#999">-</td>'
        t = max(0.0, min(1.0, float(ks) / 100))
        g = int(180 + 75 * t); rr = int(255 - 120 * t); bb = int(255 - 150 * t)
        return (f'<td style="text-align:center;font-weight:700;font-size:1.1em;'
                f'background:rgb({rr},{g},{bb});padding:6px 12px">{float(ks):.1f}</td>')
    n = len(df)
    th = "".join(f"<th style='padding:6px 10px'>{name}</th>" for _, name, _ in METRICS)
    trs = []
    for _, r in df.iterrows():
        uma = r.get("umaban")
        umas = f"{int(uma)}" if uma == uma and uma is not None else "？"
        tds = "".join(cell(r.get(k), r.get(k + "_rank"), d, n) for k, _, d in METRICS)
        trs.append(f"<tr><td style='text-align:center;font-weight:700;padding:6px 10px'>{umas}</td>"
                   f"{kcell(r.get('kehai_score'))}{tds}</tr>")
    html = f"""<!doctype html><meta charset="utf-8">
<title>全頭 歩様比較（気配スコア）</title>
<style>body{{font-family:sans-serif;margin:24px}}table{{border-collapse:collapse}}
th,td{{border:1px solid #ddd}}th{{background:#333;color:#fff}}</style>
<h2>全頭 歩様指標 横並び比較（気配スコア順）</h2>
<table><thead><tr><th style='padding:6px 10px'>馬番</th>
<th style='padding:6px 10px'>気配<br>スコア</th>{th}</tr></thead>
<tbody>{''.join(trs)}</tbody></table>
<p>気配スコア(0〜100・高いほど良い)＝背の安定・頭の落ち着き・ストライドの伸び・リズムを
良し悪しの向きで合成した相対指標。濃い緑＝その指標で相対的に好ましい側 / ★＝field内トップ。
<b>相対比較であり勝敗の断定ではありません。</b>骨格付き動画での目視確認と併用してください。</p>"""
    with open(path, "w") as f:
        f.write(html)
    print(f"HTML表を保存: {path}")


def selftest():
    import pandas as pd
    rows = [
        {"umaban": 3, "stride_freq_hz": 0.80, "stride_regularity_s": 0.05,
         "stride_len_norm": 0.50, "head_bob_norm": 0.03, "topline_stability": 0.02},
        {"umaban": 7, "stride_freq_hz": 0.90, "stride_regularity_s": 0.20,
         "stride_len_norm": 0.40, "head_bob_norm": 0.10, "topline_stability": 0.08},
        {"umaban": 12, "stride_freq_hz": 0.85, "stride_regularity_s": 0.12,
         "stride_len_norm": 0.45, "head_bob_norm": 0.06, "topline_stability": 0.05},
    ]
    df = add_kehai(rank_field(rows))
    # 3番はregularity/head_bob/toplineが最小=好ましい向きで1位のはず
    idx = df.set_index("umaban")
    assert idx.loc[3, "stride_regularity_s_rank"] == 1
    assert idx.loc[3, "topline_stability_rank"] == 1
    # 3番は全指標で好ましい向き→気配スコア最上位のはず
    assert idx["kehai_score"].idxmax() == 3, idx["kehai_score"]
    print_table(df)
    print("SELFTEST PASSED (相対順位付け＋気配スコアが正しく動作)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("patterns", nargs="*", default=None,
                    help="CSVのglob（既定: ./*_gait_metrics.csv）")
    ap.add_argument("--html", help="HTML比較表の出力先")
    ap.add_argument("--selftest", action="store_true")
    args = ap.parse_args()
    if args.selftest:
        selftest(); return
    patterns = args.patterns or ["*_gait_metrics.csv"]
    rows = load_rows(patterns)
    if not rows:
        ap.error(f"歩様指標CSVが見つかりません: {patterns}")
    df = add_kehai(rank_field(rows))
    print_table(df)
    if args.html:
        write_html(df, args.html)
    import pandas as pd
    out = "field_gait_compare.csv"
    df.drop(columns=[c for c in df.columns if c.startswith("_")], errors="ignore").to_csv(out, index=False)
    print(f"比較CSVを保存: {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_paddock.py
#!/usr/bin/env python3
"""
run_paddock.py — パドック動画1本から「全頭の気配ランキング＋骨格動画」を一発生成

内部で4ツールを順に呼ぶ:
  1. paddock_segment.py … 馬番テロップOCRで全頭に自動分割＋各馬クリップ切り出し
  2. paddock_gait.py    … 各馬の歩様を数値化(目的馬ロック付き)
  3. skeleton_overlay.py… 各馬の骨格線＋背骨＋重心を描いた確認動画
  4. paddock_compare.py … 全頭を気配スコアで横並び比較(HTML)

使い方:
    python run_paddock.py race.mp4
    python run_paddock.py race.mp4 --fast --fps 12 --outdir race12R
    python run_paddock.py race.mp4 --roi 258,806,120,70   # 競馬場ごとのテロップ位置
    python run_paddock.py race.mp4 --no-skeleton           # 骨格動画をスキップして高速化

出力(--outdir 既定は <動画名>_paddock/):
    <馬番>のクリップ / *_gait_metrics.csv / *_skeleton.mp4 / kehai.html(気配ランキング)
"""
import argparse, glob, os, shutil, subprocess, sys

HERE = os.path.dirname(os.path.abspath(__file__))
PY = sys.executable


def sh(cmd):
    print("  $", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, check=False).returncode


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("video", help="パドック連続映像(全頭が写った1本)")
    ap.add_argument("--roi", help="馬番ボックス座標 x,y,w,h(競馬場ごと)")
    ap.add_argument("--sample-fps", type=float, default=3.0, help="馬番OCRのサンプリング")
    ap.add_argument("--fast", action="store_true", help="軽量モデルで高速化(精度は少し落ちる)")
    ap.add_argument("--fps", type=float, help="推論前にこのfpsへ間引き(例12)")
    ap.add_argument("--no-skeleton", action="store_true", help="骨格確認動画をスキップ")
    ap.add_argument("--outdir", help="出力先(既定: <動画名>_paddock)")
    args = ap.parse_args()

    if not os.path.exists(args.video):
        ap.error(f"動画が見つかりません: {args.video}")
    outdir = args.outdir or (os.path.splitext(os.path.basename(args.video))[0] + "_paddock")
    os.makedirs(outdir, exist_ok=True)
    work = os.path.join(outdir, "race.mp4")
    shutil.copy(args.video, work)
    print(f"■ 出力先: {outdir}\n")

    # 1) 全頭に自動分割＋クリップ切り出し
    print("■ [1/4] 馬番OCRで全頭に自動分割")
    seg = [PY, os.path.join(HERE, "paddock_segment.py"), work, "--cut",
           "--sample-fps", str(args.sample_fps)]
    if args.roi:
        seg += ["--roi", args.roi]
    sh(seg)
    clips = sorted(glob.glob(os.path.join(outdir, "race_uma*.mp4")))
    if not clips:
        print("！馬番を検出できませんでした。--roi でテロップ位置を調整してください。")
        return
    print(f"  → {len(clips)}頭のクリップを検出\n")

    gait_opts = ["--no-adapt"] + (["--fast"] if args.fast else []) + \
                (["--fps", str(args.fps)] if args.fps else [])

    # 2)+3) 各馬: 歩様数値化 → 骨格確認動画
    for i, clip in enumerate(clips, 1):
        name = os.path.basename(clip)
        print(f"■ [2/4] 歩様解析 ({i}/{len(clips)}): {name}")
        sh([PY, os.path.join(HERE, "paddock_gait.py"), clip] + gait_opts)
        if not args.no_skeleton:
            print(f"■ [3/4] 骨格動画 ({i}/{len(clips)}): {name}")
            sh([PY, os.path.join(HERE, "skeleton_overlay.py"), clip])
        print()

    # 4) 全頭 気配ランキング
    print("■ [4/4] 気配スコアで全頭比較")
    html = os.path.join(outdir, "kehai.html")
    sh([PY, os.path.join(HERE, "paddock_compare.py"),
        os.path.join(outdir, "*_gait_metrics.csv"), "--html", html])

    print("\n==============================")
    print(f"完了！ 出力先: {outdir}")
    print(f"  ・気配ランキング: {html}")
    print(f"  ・各馬の骨格動画: {outdir}/*_skeleton.mp4")
    print(f"  ・各馬の歩様CSV : {outdir}/*_gait_metrics.csv")
    print("==============================")


if __name__ == "__main__":
    main()


## ③ 動画を用意（どちらか）
### 方法A（おすすめ・速い）: Googleドライブ経由
Googleドライブに `paddock` フォルダを作り動画を入れておく → 下の▶（最新動画を自動使用）

In [ ]:
from google.colab import drive
import glob, os
drive.mount('/content/drive')
cands = sorted(glob.glob('/content/drive/MyDrive/paddock/*.mp4') +
               glob.glob('/content/drive/MyDrive/paddock/*.MP4'), key=os.path.getmtime)
VIDEO = cands[-1] if cands else None
print("使う動画:", VIDEO or "→ MyDrive/paddock/ に動画を入れてください")


### 方法B: ブラウザから直接アップロード（方法Aなら不要）

In [ ]:
from google.colab import files
up = files.upload(); VIDEO = list(up.keys())[0]; print("アップロード完了:", VIDEO)


## ③.5 テロップ位置の確認（④で「馬番を検出できませんでした」と出た時だけ）
下の▶を押すと、動画の1コマがグリッド付きで表示されます。
**馬番ボックスの左上の x, y と 幅 w・高さ h をグリッドの数字で読み取り**、④の `ROI` に設定してください。
（読み取りが難しければ、この画像のスクショをClaudeに送れば、正確なROIをお伝えします）

In [ ]:
import cv2, matplotlib.pyplot as plt
cap=cv2.VideoCapture(VIDEO); cap.set(cv2.CAP_PROP_POS_FRAMES,int((cap.get(7) or 300)*0.3))
ok,fr=cap.read(); cap.release()
H,W=fr.shape[:2]; img=cv2.cvtColor(fr,cv2.COLOR_BGR2RGB)
plt.figure(figsize=(15,9)); plt.imshow(img)
plt.xticks(range(0,W,50),rotation=90); plt.yticks(range(0,H,50)); plt.grid(color='lime',lw=0.6)
plt.title(f"馬番ボックスの左上(x,y)と幅w・高さh を読み取る   W={W} H={H}")
plt.show(); print(f"動画サイズ W={W} H={H}  →  ④の ROI に 例: ROI='260,806,120,70' の形で設定")


## ④ 解析（全頭 → 気配ランキング＋骨格動画）
高速モード(`--fast --fps 12`)。GPUなら1頭数秒。
- 馬番が読めない時は ③.5 で位置を調べ、`ROI="x,y,w,h"` を設定して再実行。
- ROI を None のままにすると自動探索します（30秒ほど・低画質だと外すことあり）。

In [ ]:
import subprocess, os
from IPython.display import HTML, display
FAST=True; FPS=12; ROI=None   # ROI例: "260,806,120,70"
cmd=["python","run_paddock.py",VIDEO,"--outdir","result"]+(["--fast"] if FAST else [])+(["--fps",str(FPS)] if FPS else [])+(["--roi",ROI] if ROI else [])
print("解析中…"," ".join(cmd)); subprocess.run(cmd)
html="result/kehai.html"
if os.path.exists(html):
    print("\n★ 気配ランキング ★"); display(HTML(open(html,encoding="utf-8").read()))
else:
    print("馬番を検出できませんでした → ③.5 で位置を調べ、上の ROI を設定して④を再実行")


## ⑤ 結果をダウンロード（気配表＋骨格動画一式）

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("paddock_result","zip","result"); files.download("paddock_result.zip")
print("骨格動画は zip 内の *_skeleton.mp4（背骨=緑・重心=赤）")
